<a href="https://colab.research.google.com/github/FANGxPC/LIFELOG_AI/blob/master/LIFELOG_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install specialized versions for Qwen2-VL video support
!pip install git+https://github.com/huggingface/transformers@21fac7abba2a37fae86106f87fcf9974fd1e3830 accelerate -q
!pip install qwen-vl-utils[decord] av -q
!pip install faster-whisper -q


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
from faster_whisper import WhisperModel
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Force torch to be global
global torch

print("🚀 Initializing Qwen2-VL...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
# 3. Faster-Whisper (Small)
whisper_model = WhisperModel("small.en", device="cuda", compute_type="float16")

print("✅ Initialization Complete. Memory is tight, so keep those clips at 10s!")

🚀 Initializing Qwen2-VL...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Initialization Complete. Memory is tight, so keep those clips at 10s!


In [34]:
from google.colab.output import eval_js
import base64
def capture_10s_moment(filename):
    # We pass the filename in to ensure the main loop controls it
    js_code = f'''
    (async function() {{
      try {{
        const stream = await navigator.mediaDevices.getUserMedia({{video: true, audio: true}});
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = (e) => chunks.push(e.data);
        recorder.start();

        console.log("Recording 10s...");
        await new Promise(r => setTimeout(r, 10000));

        recorder.stop();
        return new Promise(r => {{
          recorder.onstop = () => {{
            const blob = new Blob(chunks, {{type: 'video/webm'}});
            const reader = new FileReader();
            reader.readAsDataURL(blob);
            reader.onloadend = () => r(reader.result);
            stream.getTracks().forEach(t => t.stop());
          }}
        }});
      }} catch (err) {{
        return "ERROR:" + err.name;
      }}
    }})()
    '''
    data = eval_js(js_code)

    if data.startswith("ERROR:"):
        return data # Pass the error string back to handle it in Python

    binary = base64.b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return "SUCCESS"

In [35]:
prompt_instruction = """
**STRICT GROUNDING RULES (NEVER BREAK)**:
- Output EXACTLY ONE single flowing paragraph, maximum 140 words.
- First describe ALL clearly visible objects with attributes (color, type, material, brand if visible) and their EXACT positions ("black earbuds on the table 15 cm right of the notebook", "water bottle under the laptop", "phone held in right hand", etc.).
- Then describe exactly what the user is doing (posture, hands, gaze, object interactions).
- Naturally embed the full verbatim audio transcript.
- End with exactly: " Logic-based Summary: [one short factual sentence max 25 words using audio intent + visible objects/actions]."

Do NOT invent anything. If no clear objects, say "No clear objects visible". Stay clinical and short. No headings, no line breaks.

Audio Transcript: {TRANSCRIPT_PLACEHOLDER}
"""

Checkign thigns


In [36]:
import threading
import queue
import time
import os
from IPython.display import clear_output
import torch

# 1. Setup the Queue
video_queue = queue.Queue()

def ai_processor_worker():
    """Background thread: Processes videos with the Reality Auditor Prompt."""
    print("🧠 AI Reality Auditor is online...")

    while True:
        video_path = video_queue.get()
        if video_path is None: break

        try:
            # 1. Transcribe Speech
            segments, _ = whisper_model.transcribe(video_path)
            transcript = " ".join([s.text for s in segments]).strip()
            audio_display = transcript if transcript else "[No Speech Detected]"

            # PRINT AUDIO (you wanted this)
            print(f"\n🎤 AUDIO TRANSCRIPT: {audio_display}")

            # Inject transcript into the new prompt
            full_prompt = prompt_instruction.format(TRANSCRIPT_PLACEHOLDER=audio_display)

            # 2. Prepare Messages
            messages = [{
                "role": "user",
                "content": [
                    {
                        "type": "video",
                        "video": video_path,
                        "fps": 0.5,          # ← LOWERED for T4 stability (fixes hallucination)
                    },
                    {
                        "type": "text",
                        "text": full_prompt
                    }
                ]
            }]

            # 3. Process Inputs
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt"
            ).to("cuda")

            # 4. Generate Structured JSON Pulse
                        # 4. Generate with maximum grounding & anti-repetition
            with torch.no_grad():
                gen_ids = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    temperature=0.0,
                    do_sample=False,
                    repetition_penalty=2.4,
                    top_p=1.0
                )
                output = processor.batch_decode(gen_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
            print(f"\n📡 PULSE CAPTURED [{time.strftime('%H:%M:%S')}]")
            print(output) # This is your structured JSON
            print("-" * 40)

            # 6. Memory & File Cleanup
            del inputs, gen_ids
            torch.cuda.empty_cache()
            if os.path.exists(video_path):
                os.remove(video_path)

        except Exception as e:
            print(f"❌ Auditor Error: {e}")
        finally:
            video_queue.task_done()

# 2. Start the Background Processor
worker_thread = threading.Thread(target=ai_processor_worker, daemon=True)
worker_thread.start()

# 3. Main Loop (The Continuous Recorder)
def start_infinite_parallel_lifelog():
    print("🚀 INFINITE SYNC RESTARTED")
    loop = 0

    try:
        while True:
            loop += 1
            filename = f"vid_{int(time.time())}.webm"

            # Attempt to record
            status = capture_10s_moment(filename)

            if status == "SUCCESS":
                video_queue.put(filename)
                # Keep only 20 files at a time just in case cleanup fails
                if loop % 10 == 0:
                    torch.cuda.empty_cache()
            else:
                print(f"⚠️ Camera Glitch ({status}). Retrying in 2s...")
                time.sleep(2)

    except KeyboardInterrupt:
        print("\n🛑 Loop manually stopped.")
        video_queue.put(None)

# Run this to start again
start_infinite_parallel_lifelog()

🧠 AI Reality Auditor is online...
🚀 INFINITE SYNC RESTARTED
⚠️ Camera Glitch (ERROR:NotReadableError). Retrying in 2s...
⚠️ Camera Glitch (ERROR:NotReadableError). Retrying in 2s...

🎤 AUDIO TRANSCRIPT: I think I should sell my this ear but I need to buy Lamborghini but this is only  700 rupees

📡 PULSE CAPTURED [09:32:20]
Logic-Based Summarization:

The individual appears seated indoors wearing blue clothing while gesturing towards his left side where an item labeled as 'ear' can be seen placed near some clothes or fabric items that appear out-of-focus behind him; there's also another piece located further back which seems like it could possibly belong either directly next door for privacy reasons due its proximity without any direct interaction between them - hence not being considered part 
of same scene contextually speaking – thus making these two pieces indistinguishable from each other visually when considering spatial relationships within room layout constraints such positionin